# Assignment 2: Bayesian Generalization

This is the **GenJAX (canonical)** stencil. Two paired stencils are available — `generalization_python.ipynb` (Python + numpy) and `generalization_nosoln.Rmd` (R + ggplot2). Matlab available on request.

For this assignment, you will build a **Bayesian generalization model** for six animals: Cow, Dolphin, Chicken, Seal, Penguin, and Bat. There is no single "right" hypothesis space — *you* design it from properties shared by these animals.

**The setup.** Given that you observe one or more animals have some novel property, how likely is it that the other animals also have it? The Bayesian generalization framework solves this in three steps:

1. **Hypothesis space.** A set $\mathcal{H}$ of hypothesized properties. Each hypothesis $h$ is a binary vector of length 6 (1 if the animal has the property, 0 if not).
2. **Posterior over hypotheses.** After observing animal(s) ${\bf x}$ have the property,
   $$P(h \mid {\bf x}) = \frac{P(h)\,\prod_n P(x_n \mid h)}{\sum_{h' \in \mathcal{H}} P(h')\,\prod_n P(x_n \mid h')}.$$
3. **Predictive distribution over animals.** For each unobserved animal $y$,
   $$P(y \text{ has property} \mid {\bf x}) = \sum_{h:\, y \in h} P(h \mid {\bf x}).$$

We compare two likelihoods:
- **Weak sampling:** $P(x \mid h) = 1$ if $x \in h$, else $0$.
- **Strong sampling:** $P(x \mid h) = 1/|h|$ if $x \in h$, else $0$ (where $|h|$ is the number of animals in $h$).

**Corresponding textbook chapter:** [Tutorial 3 Ch 6 — Generalization](https://josephausterweil.github.io/probintro/intro2/06_generalization/) (and revisit T1 Ch 5 on Bayesian inference).


## Setup

Run the install cell if you are in Google Colab (uncomment the `!pip` line). Then run the imports.

**Dtype note.** GenJAX distributions produce `float32`. Any Python/numpy scalar passed into a `@gen` model or a `ChoiceMap` must be cast with `jnp.float32(...)` (or `jnp.int32(...)` for discrete latents) — mixing in numpy `float64` raises a `TypeError` inside the sampler.


In [ ]:
# Run on first launch in Colab:
# !pip install genjax


In [ ]:
import jax
import jax.numpy as jnp
import jax.random as random
from genjax import gen, categorical
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

np.random.seed(42)
key = random.PRNGKey(42)

ANIMALS = ("cow", "dolphin", "chicken", "seal", "penguin", "bat")
N_ANIMALS = len(ANIMALS)
print(f"Animals (in fixed order): {ANIMALS}")


---

## Problem 1: Define your hypothesis space

Write down your hypotheses. Each hypothesis is a binary vector of length 6 (one entry per animal in the order Cow, Dolphin, Chicken, Seal, Penguin, Bat). Give each hypothesis a 1–4 word label (e.g. "has wings", "lives in water").

**Constraints:**
- Include a **catch-all** hypothesis containing all six animals.
- Use **more than 4** and **fewer than 63** hypotheses.
- Each entry is 0 or 1 — no animal is "partially" in a hypothesis.

There is no single correct hypothesis space. Pick properties you think are meaningful for these animals.


### Building $\mathcal{H}$ as a matrix

Represent the hypothesis space as a 2-D `jnp.array` of shape `(H, 6)` where `H` is the number of hypotheses and each row is a hypothesis vector. Keep a parallel list of 1–4 word labels in the **same order** as the rows.

**Fill me.** Define `hypothesis_labels` and `hypothesis_matrix` below. Below the cell, briefly describe the properties you chose.

In [ ]:
# fill me
#
# Suggested approach:
#   1. List your hypothesis labels (strings, 1-4 words each).
#   2. For each label, write a length-6 binary vector — in the SAME order as
#      ANIMALS = ("cow", "dolphin", "chicken", "seal", "penguin", "bat").
#   3. Stack them into a single matrix.
#   4. Don't forget the catch-all (all ones).
#   5. Keep `H > 4` and `H < 63`.
#
# Example shape (replace with your own):
#   hypothesis_labels = ["catch-all (any animal)", "lives in water", ...]
#   hypothesis_matrix = jnp.array([
#       [1, 1, 1, 1, 1, 1],   # catch-all
#       [0, 1, 0, 1, 1, 0],   # lives in water
#       ...
#   ], dtype=jnp.float32)

hypothesis_labels = [
    # your labels here
]

hypothesis_matrix = jnp.array(
    [
        # your binary rows here
    ],
    dtype=jnp.float32,
)

H = hypothesis_matrix.shape[0]
assert hypothesis_matrix.shape == (H, N_ANIMALS), f"expected (H, {N_ANIMALS}), got {hypothesis_matrix.shape}"
assert len(hypothesis_labels) == H, "labels and matrix rows must align"
assert 4 < H < 63, "use more than 4 and fewer than 63 hypotheses"
assert jnp.all((hypothesis_matrix.sum(axis=1) > 0)), "no empty hypotheses"
assert jnp.any(jnp.all(hypothesis_matrix == 1, axis=1)), "include a catch-all hypothesis"
print(f"H = {H} hypotheses, sizes = {hypothesis_matrix.sum(axis=1).astype(int).tolist()}")


*(Describe your hypotheses here in 1–2 sentences — what kinds of properties did you include?)*

---

## Problem 2: Prior

Define a prior $P(h)$ over your hypotheses. A uniform prior (every hypothesis equally likely) is fine. Write 1–2 sentences justifying your choice.


In [ ]:
# fill me
#
# Suggested approach:
#   1. A uniform prior is `jnp.full((H,), 1.0 / H)`.
#   2. If you prefer a non-uniform prior, build any positive vector of length H and normalize.
#   3. Make sure `prior` sums to 1.

prior = None   # replace with a length-H jnp array that sums to 1
assert prior is not None and prior.shape == (H,), "prior must have shape (H,)"
assert jnp.isclose(prior.sum(), 1.0), "prior must sum to 1"


*(1–2 sentences justifying your prior.)*

---

## Problem 3: Posterior

Compute the posterior $P(h \mid {\bf x})$ under both **weak** and **strong** sampling. Write the weak/strong likelihood functions, multiply by the prior, normalize.


### The generative model as a `@gen` function

The Bayesian-generalization model is **discrete in `h`**: there are finitely many hypotheses, so we can enumerate them. We will:

1. Write the generative process as a `@gen` function — `h_idx` sampled from `categorical(log_prior)`, then each observed animal $x$ generated according to the chosen sampling rule given `h`.
2. Compute the posterior over `h_idx` by **enumeration** — evaluate the unnormalized posterior at every hypothesis and normalize. This is exact (no sampling needed) because the latent is discrete with a small support.

**`categorical(logits)` in GenJAX takes log-probabilities** (logits), so use `jnp.log(prior)`. It returns an integer index in `[0, H)`.

**`@gen` cheatsheet (Tutorial 2 Ch 2–4).**
- `x = dist(args) @ "name"` — sample from a distribution, address the choice as `"name"`.
- `model.simulate(key, args)` — run the model once forward, get a trace.
- `model.assess(choices, args)` — return `(log_density, retval)` for an entire trace of choices. We use this for enumeration.

**Fill me** — write the model and the per-hypothesis log-likelihood. The model itself is short; the enumeration logic does the real work below.

In [ ]:
# fill me — log-likelihood of observations under hypothesis h
#
# Given:
#   h:        length-N_ANIMALS binary vector
#   x_idxs:   indices (into ANIMALS) of the observed animals
#   sampling: "weak" or "strong"
#
# Suggested approach (weak):
#   - log P(x | h) = 0  if h[x] == 1  else -inf  (i.e. likelihood 1 or 0)
#   - For multiple x's, sum the per-x log-likelihoods.
#
# Suggested approach (strong):
#   - log P(x | h) = -log |h|  if h[x] == 1  else -inf
#   - For multiple x's, sum (since data are iid given h).
#
# Use jnp.where + jnp.log and -jnp.inf (or a large negative number like -1e9) to
# represent the impossible outcomes.

def log_likelihood(h, x_idxs, sampling):
    """Return log P(x_idxs | h) under weak or strong sampling.

    Args:
        h:        jnp.array shape (N_ANIMALS,) of 0/1 (float ok)
        x_idxs:   jnp.array of integer indices into ANIMALS, shape (n_obs,)
        sampling: "weak" or "strong"
    """
    # fill me
    pass


In [ ]:
# fill me — vectorized posterior over hypotheses by enumeration
#
# Suggested approach:
#   1. For each row h_i in hypothesis_matrix, compute
#         log_post_unnorm[i] = log(prior[i]) + log_likelihood(h_i, x_idxs, sampling)
#   2. Stabilize with log-sum-exp:
#         m = log_post_unnorm.max()
#         post = jnp.exp(log_post_unnorm - m)
#         post = post / post.sum()
#   3. Vectorize the per-row computation with jax.vmap.

def posterior(x_idxs, sampling, hyp_matrix=hypothesis_matrix, prior_=prior):
    """Return posterior P(h | x_idxs) as a length-H jnp.array that sums to 1."""
    # fill me
    pass


# Now also wrap the same model as a @gen function so you can simulate from it.
# This is short — the heavy lifting was the enumeration above.

@gen
def generalization_model(log_prior, hyp_matrix, x_idxs_placeholder, sampling):
    """A generative model for a single observation x_1, given hypothesis space."""
    # fill me
    #
    # Suggested approach:
    #   1. Sample h_idx = categorical(log_prior) @ "h_idx".
    #   2. Pick h = hyp_matrix[h_idx].
    #   3. (Optional) sample the observed animal x_1 from the appropriate
    #      distribution under weak/strong. For weak: any animal in h equiprobable.
    #      For strong: same — uniform over animals in h. (Weak vs. strong differ
    #      only in the LIKELIHOOD, not in forward sampling — both sample uniformly
    #      from h. The difference shows up when you condition on x_1.)
    #   4. Return h_idx.
    pass


### 3(a): One observation

Pick one animal and compute the posterior under weak and strong sampling. Plot both as a bar chart over your hypothesis labels.

**Write 1–2 sentences:** how does the posterior change after observing one animal? Are there differences between weak and strong sampling? If so, what are they (and why)?


In [ ]:
# fill me
#
# 1. Pick ONE observed animal (an index into ANIMALS, say 2 for "chicken").
# 2. Call posterior(...) for both samplings.
# 3. Bar-plot both, side by side, with hypothesis_labels on the x-axis.

one_obs_idx = jnp.array([2])   # e.g. chicken; pick any animal you want

post_weak_1 = None             # replace with posterior(one_obs_idx, "weak")
post_strong_1 = None           # replace with posterior(one_obs_idx, "strong")

# Plot:
# fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
# x = np.arange(H)
# axes[0].bar(x, np.array(post_weak_1));   axes[0].set_xticks(x); axes[0].set_xticklabels(hypothesis_labels, rotation=45, ha='right'); axes[0].set_title(...)
# axes[1].bar(x, np.array(post_strong_1)); axes[1].set_xticks(x); axes[1].set_xticklabels(hypothesis_labels, rotation=45, ha='right'); axes[1].set_title(...)
# plt.tight_layout(); plt.show()

# your plotting code here


**Your answer (3a).** *(1–2 sentences. How did the posterior change? Differences between weak and strong sampling, and why?)*

### 3(b): Three observations

Add two more animals (so three animals total) and recompute the posterior under both samplings. Plot.

**Write 1–2 sentences:** how has the posterior changed compared to (a)? How does that differ between weak and strong sampling?


In [ ]:
# fill me
#
# 1. Pick three observed animals (3 distinct indices).
# 2. posterior(three_obs_idxs, "weak") / posterior(three_obs_idxs, "strong").
# 3. Plot — same style as 3(a).

three_obs_idxs = jnp.array([2, 5, 4])   # e.g. chicken, bat, penguin

post_weak_3 = None
post_strong_3 = None

# your plotting code here


**Your answer (3b).** *(1–2 sentences. Compared to 3(a) — what changed? Weak vs. strong?)*

---

## Problem 4: Predictive distribution

For each of the six animals $y$, compute
$$P(y \text{ has property} \mid {\bf x}) = \sum_{h:\, y \in h} P(h \mid {\bf x}).$$

Make **four** histograms (1-obs weak, 1-obs strong, 3-obs weak, 3-obs strong) — one bar per animal, height = predictive probability. Label axes and title each plot.

**Write a paragraph** describing the results: which animals are predicted to share the property? How does this differ between weak and strong, and between 1 vs. 3 observations? Tie the patterns back to the hypotheses that survived in the posterior.


In [ ]:
# fill me — predictive distribution over animals
#
# Suggested approach:
#   For each animal y in 0..N_ANIMALS-1:
#     P(y has property | x) = sum over h such that h[y] == 1 of posterior[h]
#   This is the matrix product:  posterior @ hypothesis_matrix   (shape (H,) @ (H, N_ANIMALS) -> (N_ANIMALS,))

def predictive(post, hyp_matrix=hypothesis_matrix):
    """Return length-N_ANIMALS vector of P(y has property | observations)."""
    # fill me
    pass


# Then build the four histograms:
#   pred_weak_1   = predictive(post_weak_1)
#   pred_strong_1 = predictive(post_strong_1)
#   pred_weak_3   = predictive(post_weak_3)
#   pred_strong_3 = predictive(post_strong_3)
#
# A 2x2 grid of bar plots (rows = #observations, cols = sampling) reads well.

# your code + plotting here


**Your answer (Problem 4).** *(A paragraph. Which animals get high predictive probability, under each condition? Tie back to which hypotheses survived in the posterior. How does weak vs. strong differ?)*

---

## Problem 5: Break your model

Now expand the hypothesis space to **all** $2^6 - 1 = 63$ non-empty binary vectors of length 6 (i.e. every possible subset of the six animals except the empty set). Use a uniform prior over this expanded space. Recompute the posterior and predictive distributions from Problems 3 and 4.


In [ ]:
# fill me — enumerate all 63 non-empty subsets of 6 animals
#
# Suggested approach:
#   1. There are 2^6 = 64 binary vectors of length 6. Drop the all-zeros one.
#   2. itertools.product([0, 1], repeat=6) generates all 64.
#   3. Stack into a jnp.array, drop the zero row, dtype=float32.
#   4. Uniform prior over all 63.
#   5. Reuse posterior() and predictive() — they only need a hypothesis matrix and prior.

import itertools

all_hyp_matrix = None     # shape (63, 6)
all_prior = None          # shape (63,) uniform

# your code here


In [ ]:
# fill me — repeat 3(a) / 3(b) / Problem 4 with the expanded hypothesis space.
#
# You don't need ALL 63 individual posterior values in your write-up — pick the
# slices that make your point (e.g. the predictive bars; the most-likely 5
# hypotheses; the entropy of the predictive).

# Example calls (you'll need to thread the new (hyp_matrix, prior_) through your posterior/predictive):
#   post_weak_1_big   = posterior(one_obs_idx,    "weak",   hyp_matrix=all_hyp_matrix, prior_=all_prior)
#   post_strong_1_big = posterior(one_obs_idx,    "strong", hyp_matrix=all_hyp_matrix, prior_=all_prior)
#   pred_weak_1_big   = predictive(post_weak_1_big, hyp_matrix=all_hyp_matrix)
#   ... etc for the 3-obs case.

# your code + plotting here


**Write a few sentences:** what happened to the posterior and predictive probabilities? Why? **Relate this to a theorem we covered in class.** (Hint: think about what a uniform prior over *all possible* hypotheses encodes about your beliefs — and what it does *not* encode.)

You do not need to dump all 63 hypotheses' probabilities in your report — just enough to make your point.


**Your answer (Problem 5).** *(A few sentences — what happened to the posterior and predictive? Why? Which theorem from class does this illustrate?)*

---

## Submission

Submit by DM or email to the instructor **one** of:

- your completed notebook — it must run end-to-end with no errors and contain your figures, inline text answers, and descriptions; **or**
- a single PDF report containing your code, figures, text answers, and descriptions.
